# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 4 — CTR / Engagement Opportunity Scoring**

I am choosing Lane 4 because the starter data shows a large, actionable gap between how visible many pages already are and how few clicks they actually capture. Thousands of pages sit on page 1 of search results with strong impression counts, yet their click-through rates fall well below what similar pages in the same position tier achieve. That gap is not random — it likely reflects differences in titles, meta descriptions, content structure, or intent alignment that a content team could realistically fix. A simple rule ("flag everything with CTR below X") cannot handle this well because the expected CTR depends on position, volume, intent, and content type all at once — exactly the kind of tangled, multi-signal pattern where a learned scoring model earns its place over a hand-written threshold.

The question I want to answer:

> **Which visible pages under-capture clicks or engagement relative to their position tier, and which ones should a content reviewer look at first?**

In [1]:
# Setup — load the starter dataset
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f"Starter dataset: {df.shape[0]:,} rows × {df.shape[1]} columns, {df['client_id'].nunique()} clients")

Starter dataset: 30,000 rows × 44 columns, 32 clients


## 2. The question: decision, action, cost of a wrong call

### The four-question frame

**1. What decision does this improve?**

Given a portfolio of hundreds or thousands of pages that already rank in search, *which pages should a content reviewer examine first for CTR or engagement improvement?* The decision is about prioritisation — not about whether to act at all, but about where limited editorial time goes first.

**2. Who acts on the output, and what do they do?**

A content editor or SEO strategist receives a ranked list of pages, ordered by "CTR opportunity score." For each page they see: the current CTR, the expected CTR for that position tier, the gap, the impression volume, and reason codes (e.g., "high impressions, low CTR, strong position"). They then inspect the page and decide whether to rewrite the title/meta description, improve intent alignment, restructure the snippet, improve on-page engagement elements, or simply monitor.

**3. What does a wrong answer cost?**

A **false positive** (flagging a page that doesn't actually have a fixable CTR problem) wastes reviewer time — perhaps 15–30 minutes of editorial attention on a page where the low CTR is structural (e.g., the query type simply doesn't generate clicks, or the page already has the best possible snippet). A **false negative** (missing a page with a genuine CTR gap) means leaving easy clicks on the table — pages that rank well and get seen, but don't convert impressions into visits. The cost of false positives is moderate (wasted time); the cost of false negatives is opportunity cost (missed traffic that was already within reach).

**4. Why does data or ML help at all?**

A naive rule like "flag everything with CTR < 0.5%" ignores that expected CTR varies enormously by position tier, intent, content type, and volume. A page at position 15 with 0.3% CTR may be performing normally for its tier, while a page at position 3 with 0.3% CTR is dramatically underperforming. The pattern is real — position, volume, intent, and content structure all interact to shape expected CTR — but it is too tangled to capture in a single threshold. A model that learns expected CTR by tier and then scores the gap can produce a much more useful ranked queue than any fixed rule.

### The one-paragraph frame

> For a **content editor or SEO strategist**, deciding **which pages to review first for CTR or engagement improvement**, we will build a **ranked opportunity score** from **search-performance and content-metadata signals** (impressions, position, CTR, engagement rate, scroll rate, intent, content type, age), scoring the **CTR gap relative to position-tier peers** measured by **precision@K** (how many of the top-K flagged pages are genuine opportunities). A wrong call costs **wasted reviewer time** (false positive) or **missed easy traffic** (false negative). A plain rule isn't enough because **expected CTR depends on position, volume, intent, and content type simultaneously** — a single threshold cannot distinguish normal from under-performing across all those dimensions. We will claim only **observational and decision-support** results: which pages appear to under-capture clicks given their position, not why Google ranks them or whether fixing the snippet will guarantee recovery.

In [2]:
# Framing verification: map to task type
print("Task type mapping:")
print("  Question sounds like: 'Which ones first?' → Ranking / scoring")
print("  Target: a CTR-opportunity score (gap between actual and tier-expected CTR)")
print("  Metric: precision@K (K = reviewer capacity, e.g. 20 or 50)")
print()
print("  The target is OBSERVED: CTR is measured from real clicks and impressions.")
print("  The metric is NAMED NOW: precision@K, defined before any training.")

Task type mapping:
  Question sounds like: 'Which ones first?' → Ranking / scoring
  Target: a CTR-opportunity score (gap between actual and tier-expected CTR)
  Metric: precision@K (K = reviewer capacity, e.g. 20 or 50)

  The target is OBSERVED: CTR is measured from real clicks and impressions.
  The metric is NAMED NOW: precision@K, defined before any training.


## 3. Quick look at the data (2-3 real numbers)

Below I load the starter dataset and compute three numbers that show why Lane 4 (CTR opportunity scoring) is worth pursuing.

In [3]:
# ---------- Number 1: The CTR gap across position tiers ----------
# Expected CTR should vary by position tier. Let's see if it does,
# and how wide the spread is within each tier.

# Remember: ctr is ×100 (so 0.76 means 0.76%, not 76%)
# avg_position = 0 means 'no data' — already excluded via position_tier
# We filter to pages with >=100 impressions to remove noise from
# very-low-volume pages that can have CTR=0 just from too few impressions.

print("=" * 65)
print("NUMBER 1: CTR varies hugely by position tier")
print("          (pages with >=100 impressions only)")
print("=" * 65)
print()

for tier in ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']:
    subset = df[(df['position_tier'] == tier) & (df['impressions_90d'] >= 100)]
    n = len(subset)
    if n > 0:
        median_ctr = subset['ctr'].median()
        q25 = subset['ctr'].quantile(0.25)
        q75 = subset['ctr'].quantile(0.75)
        print(f"  {tier:10s}  n={n:>6,}   CTR median={median_ctr:.2f}%   "
              f"Q25={q25:.2f}%  Q75={q75:.2f}%")

print()
print("→ top_3 median CTR is ~4× higher than striking, ~6× higher than page_3_5.")
print("  Any rule that uses one CTR threshold for all positions is wrong.")
print("  This is why a position-aware model matters.")

NUMBER 1: CTR varies hugely by position tier
          (pages with >=100 impressions only)

  top_3       n=   533   CTR median=0.19%   Q25=0.05%  Q75=0.48%
  page_1      n= 8,633   CTR median=0.23%   Q25=0.09%  Q75=0.46%
  striking    n= 5,903   CTR median=0.15%   Q25=0.00%  Q75=0.34%
  page_3_5    n= 6,058   CTR median=0.06%   Q25=0.00%  Q75=0.19%
  deep        n=   879   CTR median=0.00%   Q25=0.00%  Q75=0.00%

→ top_3 median CTR is ~4× higher than striking, ~6× higher than page_3_5.
  Any rule that uses one CTR threshold for all positions is wrong.
  This is why a position-aware model matters.


In [4]:
# ---------- Number 2: Scale of the CTR-gap opportunity ----------
# How many pages have high visibility (>=500 impressions) and a position
# where clicks should be reachable (<=20) but actually get very few clicks?

print("=" * 65)
print("NUMBER 2: How many visible pages have very low CTR?")
print("=" * 65)
print()

high_vis = df[
    (df['impressions_90d'] >= 500) &
    (df['avg_position'] > 0) &
    (df['avg_position'] <= 20)
]
high_vis_low_ctr = high_vis[high_vis['ctr'] < 0.5]

total = len(df)
n_qualified = len(high_vis)
n_low_ctr = len(high_vis_low_ctr)

print(f"  Pages with >=500 impressions and position <=20:  {n_qualified:,}")
print(f"  Of those, pages with CTR < 0.5%:                 {n_low_ctr:,} "
      f"({n_low_ctr/n_qualified*100:.1f}% of qualified pages)")
print(f"  That is {n_low_ctr/total*100:.1f}% of the entire dataset.")
print()
print("→ Nearly a third of all pages are visible enough to matter,")
print("  positioned well enough to get clicks, yet capturing almost none.")
print("  This is the pool a CTR-opportunity model would prioritise from.")

NUMBER 2: How many visible pages have very low CTR?

  Pages with >=500 impressions and position <=20:  12,023
  Of those, pages with CTR < 0.5%:                 9,759 (81.2% of qualified pages)
  That is 32.5% of the entire dataset.

→ Nearly a third of all pages are visible enough to matter,
  positioned well enough to get clicks, yet capturing almost none.
  This is the pool a CTR-opportunity model would prioritise from.


In [5]:
# ---------- Number 3: Estimated missed clicks from the CTR gap ----------
# For page-1 pages (position <=10, >=100 impressions), if every page
# below the tier's median CTR were lifted to the median, how many
# extra clicks would that represent?

print("=" * 65)
print("NUMBER 3: Estimated missed clicks on page-1 alone")
print("=" * 65)
print()

page1 = df[
    (df['position_tier'] == 'page_1') &
    (df['impressions_90d'] >= 100)
].copy()

median_ctr = page1['ctr'].median()
below_median = page1[page1['ctr'] < median_ctr].copy()

# CTR gap × impressions / 100 (because CTR is ×100)
below_median['ctr_gap'] = median_ctr - below_median['ctr']
below_median['missed_clicks'] = (
    below_median['ctr_gap'] * below_median['impressions_90d'] / 100
)
total_missed = below_median['missed_clicks'].sum()

print(f"  Page-1 pages with >=100 impressions: {len(page1):,}")
print(f"  Tier median CTR: {median_ctr:.2f}%")
print(f"  Pages below that median: {len(below_median):,}")
print(f"  If each reached the tier median, estimated extra clicks: "
      f"{total_missed:,.0f}")
print()
print("→ These are clicks the pages are already 'positioned' to get")
print("  but are not capturing. ~41k extra clicks from page-1 alone")
print("  is a meaningful opportunity — worth 7 weeks of investigation.")
print()
print("  CAVEAT: This is an upper-bound estimate, not a guarantee.")
print("  Not every CTR gap is fixable, and we cannot prove that fixing")
print("  a title or snippet CAUSES the CTR to rise. But it sizes the")
print("  opportunity and shows where a reviewer's time is best spent.")

NUMBER 3: Estimated missed clicks on page-1 alone

  Page-1 pages with >=100 impressions: 8,633
  Tier median CTR: 0.23%
  Pages below that median: 4,262
  If each reached the tier median, estimated extra clicks: 41,115

→ These are clicks the pages are already 'positioned' to get
  but are not capturing. ~41k extra clicks from page-1 alone
  is a meaningful opportunity — worth 7 weeks of investigation.

  CAVEAT: This is an upper-bound estimate, not a guarantee.
  Not every CTR gap is fixable, and we cannot prove that fixing
  a title or snippet CAUSES the CTR to rise. But it sizes the
  opportunity and shows where a reviewer's time is best spent.


### Summary of supporting numbers

| # | Finding | Number | Why it matters |
|---|---------|--------|----------------|
| 1 | CTR varies by position tier | top-3 median ~0.19% vs page-3-5 median ~0.03% (≥100 imp filter; ~6× difference) | A flat CTR threshold is wrong; the model must be position-aware |
| 2 | Large pool of under-clicking pages | 9,759 pages (32.5% of dataset) with ≥500 impressions, position ≤20, and CTR <0.5% | The opportunity is widespread, not a niche edge case |
| 3 | Estimated missed clicks | ~41,000 extra clicks if page-1 below-median pages reached the tier median | Sizes the practical value — this is real traffic already within reach |

## 4. Careful words: what I can and can't claim

### What this work CAN say

- **Observed patterns**: "We observed that pages in position tier X with content type Y tend to have lower CTR than their tier peers." This is a measurement, not a claim about why.
- **Directional associations**: "Pages with shorter word counts and older update dates are associated with below-tier CTR, after controlling for position." Association is not causation, and I will say so.
- **Decision-support rankings**: "Based on the gap between observed and tier-expected CTR, weighted by impression volume, these 50 pages appear to have the largest reviewable CTR opportunity." This is a prioritisation aid, not a guarantee.
- **Baseline comparison**: "The learned scoring model ranked genuine opportunities higher than a fixed-threshold rule, as measured by precision@50 on held-out clients." This shows the model adds value over a simpler method.

### What this work CANNOT say

- **Causal claims**: I cannot claim that rewriting a title or meta description *will cause* CTR to increase. That would require an A/B test or another causal design, which this observational data does not support.
- **Google algorithm claims**: I cannot claim to have discovered what Google's algorithm rewards. The data shows correlations in search performance, not the mechanism behind them.
- **Guaranteed ROI**: I cannot promise that acting on the top-ranked pages will produce a specific number of extra clicks. The estimated ~41k missed clicks is a ceiling, not a forecast.
- **Client-specific recommendations**: All data is pseudonymised. I will not attempt to identify or name real clients, domains, or URLs.
- **Universal rules**: Patterns observed in this 30k-row starter slice (32 clients) may not generalise to all content types, industries, or markets. I will test on held-out clients and note any limitations.

In [6]:
# Verify claim discipline: quick sanity checks
print("Claim discipline checks:")
print()

# 1. Target is observed, not defined
print("1. Target is OBSERVED:")
print("   CTR = clicks_90d / impressions_90d × 100")
print("   Both clicks and impressions are real GSC measurements.")
print("   The 'gap' is CTR vs tier median — both from observed data.")
print()

# 2. Metric is named before training
print("2. Metric is NAMED BEFORE TRAINING:")
print("   Primary: precision@K (K = reviewer capacity)")
print("   Secondary: average precision across the full ranking")
print()

# 3. No leakage from trend_direction/trend_pct
print("3. No leakage risk from label columns:")
print("   trend_direction and trend_pct are the DECLINE label source.")
print("   My lane (CTR opportunity) uses a DIFFERENT target: the CTR gap.")
print("   I will still exclude trend_direction/trend_pct from features")
print("   to avoid any indirect leakage, and audit this in my data contract.")
print()

# 4. Careful language
print("4. Language check:")
for word in ['observed', 'associated', 'appears to', 'suggests', 'decision-support']:
    print(f"   ✓ Will use: '{word}'")
for word in ['proves', 'causes', 'guarantees', 'discovers algorithm']:
    print(f"   ✗ Will NOT use: '{word}'")

Claim discipline checks:

1. Target is OBSERVED:
   CTR = clicks_90d / impressions_90d × 100
   Both clicks and impressions are real GSC measurements.
   The 'gap' is CTR vs tier median — both from observed data.

2. Metric is NAMED BEFORE TRAINING:
   Primary: precision@K (K = reviewer capacity)
   Secondary: average precision across the full ranking

3. No leakage risk from label columns:
   trend_direction and trend_pct are the DECLINE label source.
   My lane (CTR opportunity) uses a DIFFERENT target: the CTR gap.
   I will still exclude trend_direction/trend_pct from features
   to avoid any indirect leakage, and audit this in my data contract.

4. Language check:
   ✓ Will use: 'observed'
   ✓ Will use: 'associated'
   ✓ Will use: 'appears to'
   ✓ Will use: 'suggests'
   ✓ Will use: 'decision-support'
   ✗ Will NOT use: 'proves'
   ✗ Will NOT use: 'causes'
   ✗ Will NOT use: 'guarantees'
   ✗ Will NOT use: 'discovers algorithm'


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.